# taskiq_nats

## Example 1: a core-NATS broker

In [ ]:
# Example 1: NatsBroker built from a server list, with a queue group, without connecting
from taskiq import AsyncBroker
from taskiq_nats import NatsBroker

broker = NatsBroker(["nats://127.0.0.1:4222", "nats://127.0.0.1:4223"], subject="jobs", queue="workers")
assert isinstance(broker, AsyncBroker)
assert broker.subject == "jobs" and broker.queue == "workers"
print("Example 1:", type(broker).__name__, broker.subject, broker.queue)

## Example 2: the two JetStream brokers

In [ ]:
# Example 2: push- and pull-based JetStream brokers carry their stream and durable consumer names
from taskiq_nats import PullBasedJetStreamBroker, PushBasedJetStreamBroker

pull = PullBasedJetStreamBroker("nats://127.0.0.1:4222", stream_name="taskiq_s", durable="taskiq_d")
push = PushBasedJetStreamBroker("nats://127.0.0.1:4222", stream_name="taskiq_s", queue="workers")
assert (pull.stream_name, pull.durable) == ("taskiq_s", "taskiq_d")
assert push.stream_name == "taskiq_s"
print("Example 2:", type(pull).__name__, type(push).__name__)

## Example 3: a task and the bytes its kick would publish

In [ ]:
# Example 3: @broker.task registers a task; the formatter shows the message published to the subject
import json
from taskiq import TaskiqMessage
from taskiq_nats import NatsBroker

broker = NatsBroker("nats://127.0.0.1:4222")

@broker.task(task_name="demo:mul")
async def mul(a: int, b: int) -> int:
    return a * b

msg = TaskiqMessage(task_id="t-2", task_name=mul.task_name, labels={}, args=[6, 7], kwargs={})
assert broker.find_task("demo:mul") is mul and json.loads(broker.formatter.dumps(msg).message)["args"] == [6, 7]
print("Example 3:", broker.formatter.dumps(msg).message[:80])

## Example 4: the object-store result backend and the key-value schedule source

In [ ]:
# Example 4: NATSObjectStoreResultBackend and NATSKeyValueScheduleSource, attached without connecting
from taskiq import TaskiqScheduler
from taskiq_nats import NatsBroker, NATSKeyValueScheduleSource, NATSObjectStoreResultBackend

backend = NATSObjectStoreResultBackend(servers="nats://127.0.0.1:4222")
broker = NatsBroker("nats://127.0.0.1:4222").with_result_backend(backend)
source = NATSKeyValueScheduleSource(servers="nats://127.0.0.1:4222")
scheduler = TaskiqScheduler(broker=broker, sources=[source])
assert broker.result_backend is backend and scheduler.sources == [source]
print("Example 4:", type(backend).__name__, type(source).__name__)

## Example 5: JetStream stream and consumer configuration

In [ ]:
# Example 5: a pull broker given its own nats-py StreamConfig and ConsumerConfig keeps them for startup
from nats.js.api import AckPolicy, ConsumerConfig, RetentionPolicy, StreamConfig
from taskiq_nats import PullBasedJetStreamBroker

stream = StreamConfig(name="jobs_stream", subjects=["jobs.>"], retention=RetentionPolicy.WORK_QUEUE, max_msgs=10_000)
consumer = ConsumerConfig(durable_name="jobs_worker", ack_policy=AckPolicy.EXPLICIT, max_deliver=3)
broker = PullBasedJetStreamBroker("nats://127.0.0.1:4222", subject="jobs.run", stream_config=stream,
                                  consumer_config=consumer, pull_consume_batch=8)
assert broker.stream_config is stream and broker.consumer_config is consumer
assert stream.as_dict()["retention"] == "workqueue" and consumer.as_dict()["ack_policy"] == "explicit"
print("Example 5:", stream.as_dict()["subjects"], consumer.as_dict()["durable_name"])